# QUBO — Kvadratna neograničena binarna optimizacija

**Quadratic Unconstrained Binary Optimization (QUBO)** je opšti matematički okvir za formulisanje optimizacionih problema koji se prirodno mapiraju na adijabatske kvantne računare i kvantnoaproksimativne algoritme (QAOA).

---

## 1. Motivacija

Mnogi realni problemi svode se na pronalaženje kombinacije vrednosti koje **minimizuju** (ili maksimizuju) neku funkciju cilja. Primeri:

- Optimizacija portfelja u finansijama
- Raspoređivanje resursa (scheduling)
- Maksimalan rez grafa (Max-Cut)
- Problem trgovačkog putnika (TSP)
- Faktorization celih brojeva

QUBO pruža **jedinstveni jezik** u koji se svi ovi problemi mogu prevesti, a koji kvantni hardver razume.

## 2. Definicija QUBO problema

### 2.1 Binarne promenljive

Radimo sa vektorom binarnih promenljivih:

$$\mathbf{x} = (x_1, x_2, \ldots, x_n), \quad x_i \in \{0, 1\}$$

Prostor pretrage ima $2^n$ mogućih rešenja.

### 2.2 Funkcija cilja

QUBO funkcija cilja je **kvadratna forma** nad binarnim promenljivama:

$$\boxed{f(\mathbf{x}) = \sum_{i} Q_{ii}\, x_i + \sum_{i < j} Q_{ij}\, x_i x_j}$$

ili kompaktno u matričnom obliku:

$$\boxed{f(\mathbf{x}) = \mathbf{x}^T Q\, \mathbf{x}}$$

gde je $Q \in \mathbb{R}^{n \times n}$ gornje-trougaona (ili simetrična) matrica koeficijenata, tzv. **QUBO matrica**.

### 2.3 Cilj

$$\mathbf{x}^* = \arg\min_{\mathbf{x} \in \{0,1\}^n} \mathbf{x}^T Q\, \mathbf{x}$$

## 3. Struktura QUBO matrice

Matrica $Q$ kodira sve informacije o problemu:

| Element | Značenje |
|---|---|
| $Q_{ii}$ (dijagonala) | Linerani doprinos promenljive $x_i$ |
| $Q_{ij}$ ($i \neq j$) | Kvadratna interakcija između $x_i$ i $x_j$ |

**Napomena:** Pošto $x_i \in \{0, 1\}$, važi $x_i^2 = x_i$, pa nema kubnih ili viših članova.

### Primer matrice za $n = 3$:

$$Q = \begin{pmatrix} Q_{11} & Q_{12} & Q_{13} \\ 0 & Q_{22} & Q_{23} \\ 0 & 0 & Q_{33} \end{pmatrix}$$

$$f(x_1, x_2, x_3) = Q_{11}x_1 + Q_{22}x_2 + Q_{33}x_3 + Q_{12}x_1x_2 + Q_{13}x_1x_3 + Q_{23}x_2x_3$$

## 4. Veza sa Izing modelom

Kvantni hardver (D-Wave, adijabatski računari) nativno radi sa **Izing (Ising) modelom**, koji koristi spin promenljive $s_i \in \{-1, +1\}$.

### Transformacija

Konverzija između binarnih i spin promenljivih:

$$x_i = \frac{1 - s_i}{2} \quad \Leftrightarrow \quad s_i = 1 - 2x_i$$

### Izing Hamiltonijan

$$H_{\text{Izing}} = -\sum_{i} h_i s_i - \sum_{i < j} J_{ij} s_i s_j$$

gde su $h_i$ lokalna polja (bias) i $J_{ij}$ kuplinzi (interakcije) između spinova.

### QUBO $\leftrightarrow$ Izing

Zamenom $x_i = \frac{1-s_i}{2}$ u $f(\mathbf{x}) = \mathbf{x}^T Q \mathbf{x}$ dobijamo Izing formu. Parametri se preslikavaju kao:

$$h_i = -\frac{1}{2}\left(Q_{ii} + \sum_{j \neq i} \frac{Q_{ij}}{2}\right), \qquad J_{ij} = -\frac{Q_{ij}}{4}$$

Ova korespondencija je ključna — **QUBO i Izing su ekvivalentni opisi istog problema**.

## 5. Konkretni primeri

### 5.1 Primer: Minimizacija jednostavne funkcije

Neka je:

$$f(x_1, x_2) = -x_1 - x_2 + 2x_1 x_2$$

QUBO matrica:

$$Q = \begin{pmatrix} -1 & 2 \\ 0 & -1 \end{pmatrix}$$

Evaluacija svih mogućih vrednosti:

In [ ]:
import numpy as np
import itertools

# QUBO matrica
Q = np.array([[-1, 2],
              [ 0,-1]])

n = Q.shape[0]
print(f"Broj promenljivih: n = {n}")
print(f"Broj mogućih rešenja: 2^n = {2**n}\n")

best_val = float('inf')
best_x = None

print(f"{'x':>12} | {'f(x)':>8}")
print("-" * 25)

for x in itertools.product([0, 1], repeat=n):
    x_vec = np.array(x)
    val = x_vec @ Q @ x_vec
    print(f"{str(x):>12} | {val:>8.2f}")
    if val < best_val:
        best_val = val
        best_x = x

print(f"\nOptimalno rešenje: x* = {best_x}")
print(f"Minimalna vrednost: f(x*) = {best_val}")

## 7. Veza sa kvantnim računarstvom

### 7.1 Adijabatsko kvantno računarstvo (AQC)

D-Wave mašine implementuju **kvantno žarenje (Quantum Annealing)**. Sistem startuje u osnovanom stanju jednostavnog Hamiltonijana $H_0$ i polako evoluira ka problemskom Hamiltonijanu $H_P$:

$$H(t) = \left(1 - \frac{t}{T}\right) H_0 + \frac{t}{T} H_P$$

Problemski Hamiltonijan direktno odgovara QUBO/Izing formulaciji:

$$H_P = \sum_i h_i \hat{\sigma}_i^z + \sum_{i<j} J_{ij} \hat{\sigma}_i^z \hat{\sigma}_j^z$$

### 7.2 QAOA (Quantum Approximate Optimization Algorithm)

QAOA je hibridni kvantno-klasični algoritam koji rešava QUBO probleme na gate-baziranim kvantnim računarima. Primenjuje naizmenično **cost** i **mixer** operatore:

$$|\boldsymbol{\gamma}, \boldsymbol{\beta}\rangle = \prod_{l=1}^{p} e^{-i\beta_l H_B} e^{-i\gamma_l H_C} |+\rangle^{\otimes n}$$

gde $H_C$ kodira QUBO funkciju cilja.

### 7.3 Zašto je QUBO centralni pojam?

```
Realni problem
      ↓
   QUBO formulacija
      ↓           ↓
 Izing model    QAOA kolo
      ↓           ↓
  D-Wave       IBM/Google
```

QUBO je **univerzalni posrednik** između klasičnih optimizacionih problema i različitih kvantnih platformi.

## 6. Ograničenja u QUBO formulaciji

QUBO je po definiciji **bez ograničenja (unconstrained)**. Međutim, mnogi realni problemi imaju ograničenja. Rešenje: **kazneni (penalty) članovi**.

### Tehnike enkodovanja ograničenja

#### Jednakosno ograničenje: $\sum_i x_i = k$

Dodajemo kaznu:

$$P\left(\sum_i x_i - k\right)^2$$

gde je $P > 0$ dovoljno velika konstanta (kazna). Razvijanjem:

$$P\left(\sum_i x_i^2 + 2\sum_{i<j} x_i x_j - 2k\sum_i x_i + k^2\right)$$

Pošto $x_i^2 = x_i$, ovo se svodi na QUBO termine.

#### Nejednakosno ograničenje: $\sum_i x_i \leq k$

Uvodimo **slack promenljive** $s_j \in \{0,1\}$:

$$\sum_i x_i + \sum_j 2^j s_j = k$$

i ovo enkodujemo kao jednakosno ograničenje.

### Izbor kazne $P$

Kazna mora biti **veća od opsega funkcije cilja** da bi narušena rešenja bila uvek lošija od validnih:

$$P > |f_{\max} - f_{\min}|$$

## 9. Rezime

| Pojam | Opis |
|---|---|
| **Promenljive** | $x_i \in \{0, 1\}$ — binarne |
| **Funkcija cilja** | $f(\mathbf{x}) = \mathbf{x}^T Q \mathbf{x}$ |
| **QUBO matrica** | $Q \in \mathbb{R}^{n \times n}$ (gornje-trougaona) |
| **Dijagonala** | Linearni doprinosi |
| **Vandijagonala** | Kvadratne interakcije |
| **Izing ekvivalent** | $s_i \in \{-1, +1\}$ — transformacijom $x_i = (1-s_i)/2$ |
| **Ograničenja** | Enkoduju se kao kazneni članovi |
| **Kompleksnost** | NP-težak u opštem slučaju |

---

> **Sledeće:** Adijabatska teorema i kvantno žarenje — kako kvantni sistem prirodno pronalazi minimum QUBO funkcije.